# Two-dimensional dijet unfolding

Build a flattened $p_{T}^{ave}$–$\eta_{CM}$ response and run a Bayesian RooUnfold closure test using the dedicated eta-dependent JER-default reconstructed distribution, response, misses, and fakes. This is a same-sample technical check of the unfolding pipeline; the independent controlled/studied notebook is the stronger closure test.

<!-- detailed-workflow-guide -->

### Detailed workflow and inverse problem

The pair $(p_T^{ave},\eta_{CM})$ is mapped to a global bin so the full response retains migrations in both coordinates. With $M_{ij}$ denoting matched events from truth bin $j$ to reco bin $i$, inclusive marginals satisfy $t_j=t_j^{matched}+t_j^{miss}$ and $m_i=m_i^{matched}+m_i^{fake}$.

Factorized unfolding applies $m_i^{signal}=P_im_i$ with $P_i=m_i^{matched}/m_i$, unfolds matched migrations with iterative Bayes, and returns inclusive truth through $\hat t_j=\hat t_j^{matched}/\epsilon_j$, $\epsilon_j=t_j^{matched}/t_j$. Forward folding reverses the physical mapping: the inclusive response reapplies efficiency and migration, then fakes are restored from the training fake-to-matched-reco ratio. Iterations regulate prior dependence versus variance; they do not guarantee unity for independent samples.

## Environment and imports

This notebook locates the repository dynamically and imports PyROOT from the
active project environment. Start Jupyter from the repository root with
`py-env/bin/python -m jupyter notebook`; no machine-specific ROOT paths are
added at runtime.

RooUnfold is loaded separately because only unfolding workflows require it.
Set `ROOUNFOLD_ROOT` when its checkout is not adjacent to this repository.


## Factorized fake and inefficiency corrections

For each reconstructed global bin, first multiply the measured spectrum by the MC purity
$P_i=N^{matched,reco}_i/N^{all,reco}_i$. Unfold this signal-only spectrum with a response built only from matched events. Finally, divide truth bin $j$ by the MC efficiency
$\epsilon_j=N^{matched,truth}_j/N^{all,truth}_j$. In symbols,
$t_j=(U[P\,m])_j/\epsilon_j$.

This factorization is valid when fake contamination is described by a reco-bin purity and loss is described by a truth-bin efficiency from representative simulation. It retains the full matched-event migration matrix. Purity and efficiency are treated as response inputs, so their finite-MC uncertainties should be evaluated with response variations or toys; they are not added again as independent bin errors here. Bins with zero efficiency cannot be recovered and stop the calculation.

### Mathematical model and flattened indexing

Let $a$ label a $p_T^{ave}$ interval and $b$ label an $\eta_{CM}$ bin. The notebook maps this pair to one global truth or reco index, $g(a,b)=aN_{\eta}+b$. This is only storage: the matrix still contains migrations between different $p_T^{ave}$ blocks and different eta bins. With $M_{ij}$ the matched-event response (reco bin $i$, truth bin $j$), the inclusive spectra obey

$$t_j^{all}=t_j^{matched}+t_j^{miss}, \qquad m_i^{all}=m_i^{matched}+m_i^{fake},$$

$$m_i^{matched}=\sum_j A_{ij}t_j^{all}, \qquad A_{ij}=M_{ij}/t_j^{all}.$$

The matrix columns therefore include efficiency when forward-folding inclusive truth. For factorized unfolding we instead normalize the migration problem with matched marginals, correct $m_i^{all}$ by $P_i=m_i^{matched}/m_i^{all}$, unfold, and divide by $\epsilon_j=t_j^{matched}/t_j^{all}$. No fake or miss is inserted into an extra matrix bin.

### What closure means here

Because pseudo-data and response use the same sample, exact arithmetic and sufficient Bayesian convergence give $\hat t_j/t_j^{all}\simeq1$. Finite precision, empty bins, and early stopping can prevent literal equality. Forward closure applies the inverse physical sequence: multiply inclusive unfolded truth by the response (which reapplies efficiency and migration), then restore the reco fake fraction. It is not performed by multiplying truth by efficiency as a separate pre-step because efficiency is already encoded in the inclusive response columns.

In [ ]:
# Cell role: initialize the reproducible Python/ROOT environment and shared helpers.
# Interpretation: No physics histogram is modified here; ROOT ownership is configured before files open.
# The preceding Markdown gives the equations and physics assumptions for this step.
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os

import sys

# Locate the repository without relying on a machine-specific absolute path.
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "CMakeLists.txt").is_file()
        and (candidate / "hist_analysis").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Cannot locate the jetAnalysis repository. Start Jupyter from its root."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root, load_roounfold

# Batch mode keeps plots reproducible and sends them to notebook/output files.
ROOT = load_root(batch=True)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import DIJET_DELTA_PHI_SELECTION_LABEL
from hist_analysis.python.histogram_io import resolve_combined_file, resolve_direction_file
from hist_analysis.python.root_style import (
    DEFAULT_PLOT_STYLE, save_canvas, set_2d_style, set_legend_style,
    set_pad_style, set_unfolding_1d_style,
)

# RooUnfold is optional and is initialized only for unfolding notebooks.
ROOUNFOLD_ROOT, ROOUNFOLD_LIBRARY = load_roounfold(
    ROOT,
    project_root=PROJECT_ROOT,
)

from hist_analysis.config.histograms import DIJET_PTAVE_BINS, TEST_DIJET_PTAVE_BINS
from hist_analysis.python.unfolding import (
    UnfoldingInputKeys, apply_efficiency_correction, as_pt_intervals,
    build_roounfold_response, calculate_closure_metrics,
    calculate_response_diagnostics, flatten_pt_eta_projections,
    flatten_sparse_response, forward_fold_truth, load_unfolding_inputs,
    project_eta_by_pt,
    prepare_factorized_corrections, project_response_eta_blocks, unfold_bayes,
    validate_response_accounting,
    unflatten_to_eta_projections, write_unfolding_output,
)
from hist_analysis.python.unfolding_plots import (
    draw_flattened_response, draw_projection_response,
    draw_response_components, draw_response_matrix,
    draw_unfolding_classification, draw_unfolding_closure,
    draw_unfolding_closure_by_pt,
)
from hist_analysis.python.plotting import draw_closure


In [ ]:
# Cell role: perform analysis step 2.
# Interpretation: Operations use the binning, normalization, and uncertainty conventions documented above.
# The preceding Markdown gives the equations and physics assumptions for this step.
# Set the ROOT style
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

## Load RooUnfold

In [ ]:
# Cell role: perform analysis step 3.
# Interpretation: Operations use the binning, normalization, and uncertainty conventions documented above.
# The preceding Markdown gives the equations and physics assumptions for this step.
try:
    import RooUnfold
except ImportError as exc:
    raise ImportError(
        f"Unable to import RooUnfold after loading {ROOUNFOLD_LIBRARY}. "
        f"Check that ROOUNFOLD_ROOT points to the RooUnfold checkout."
    ) from exc

In [ ]:
# Cell role: define and validate user-facing analysis configuration.
# Interpretation: Changing these values can change inputs, selections, binning, normalization, or outputs.
# The preceding Markdown gives the equations and physics assumptions for this step.
GENERATOR = 'embedding'       # embedding or pythia
DIRECTION = 'Pbgoing'        # pgoing, Pbgoing, or combined
FILE_STEM = 'jetId'
# List of eta cuts for analysis
ETA_CUTS = (1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.3, 2.4, 3.0)
# Finite pT-average intervals used for the 2D unfolding. Values outside this range are excluded.
PTAVE_BIN_SET = 'standard'  # test is only a quick execution check
PTAVE_BIN_SETS = {'test': TEST_DIJET_PTAVE_BINS, 'standard': DIJET_PTAVE_BINS}
PT_AVE_BINS = tuple(PTAVE_BIN_SETS[PTAVE_BIN_SET])
ETA_CUT_INDEX = 5
N_ITERATIONS = 4
COMPARISON_TARGET = 'gen'    # gen or ref; response and prior remain Gen-based
REQUIRE_FAKES = True
HANDLE_FAKES = True
FORWARD_FOLD_ADD_FAKES = True
PLOT_MISS_AND_FAKES = False
RATIO_Y_RANGE = (0.75, 1.25)
DRAW_GRID = True
MEASURED_HISTOGRAM_TEMPLATE = 'hRecoDijetPtEtaCMJerDefExtraUnfold_{eta_cut_index}'
MEASURED_LABEL = 'Reco'
# MEASURED_LABEL = 'Reco JER def.+#eta-dep.'
RESPONSE_HISTOGRAM_TEMPLATE = 'hGenDijetPtEtaCMVsRecoJerDefExtraPtEtaCM_{eta_cut_index}'
MISS_HISTOGRAM_TEMPLATE = 'hGenDijetPtEtaCMMissJerDefExtra_{eta_cut_index}'
FAKE_HISTOGRAM_TEMPLATE = 'hRecoDijetPtEtaCMFakeJerDefExtra_{eta_cut_index}'
CLASSIFICATION_HISTOGRAM_TEMPLATE = 'hUnfoldingPairClassificationJerDefExtra_{eta_cut_index}'
# Keep weighted response entries above RooUnfold's internal 1e-9 sanitization threshold.
RESPONSE_SCALE = 1.0e12
# RESPONSE_SCALE = 1.0
SAVE_PNG = False

if PTAVE_BIN_SET not in PTAVE_BIN_SETS:
    raise ValueError(f'Unsupported PTAVE_BIN_SET={PTAVE_BIN_SET!r}')
if GENERATOR not in ('embedding', 'pythia'):
    raise ValueError(f'Unsupported GENERATOR={GENERATOR!r}')
if DIRECTION not in ('pgoing', 'Pbgoing', 'combined'):
    raise ValueError(f'Unsupported DIRECTION={DIRECTION!r}')
if COMPARISON_TARGET not in ('gen', 'ref'):
    raise ValueError(f'Unsupported COMPARISON_TARGET={COMPARISON_TARGET!r}')
if ETA_CUT_INDEX < 0 or ETA_CUT_INDEX >= len(ETA_CUTS):
    raise IndexError(f'Invalid ETA_CUT_INDEX={ETA_CUT_INDEX}')
if not isinstance(PLOT_MISS_AND_FAKES, bool):
    raise TypeError('PLOT_MISS_AND_FAKES must be True or False')
if not all(isinstance(value, bool) for value in (REQUIRE_FAKES, HANDLE_FAKES, FORWARD_FOLD_ADD_FAKES)):
    raise TypeError('REQUIRE_FAKES, HANDLE_FAKES, and FORWARD_FOLD_ADD_FAKES must be booleans')
if len(RATIO_Y_RANGE) != 2 or RATIO_Y_RANGE[0] >= RATIO_Y_RANGE[1]:
    raise ValueError('RATIO_Y_RANGE must contain two increasing values')
if RESPONSE_SCALE <= 0.0:
    raise ValueError(f'RESPONSE_SCALE must be positive, got {RESPONSE_SCALE}')

if PTAVE_BIN_SET == 'test':
    print('WARNING: test pT bins contain gaps and are intended only for quick checks')

generator_label = GENERATOR.capitalize()
if DIRECTION == 'combined':
    input_path = resolve_combined_file(BASE_DIR, GENERATOR, FILE_STEM)
else:
    input_path = resolve_direction_file(BASE_DIR, GENERATOR, DIRECTION, FILE_STEM)
ETA_CUT = ETA_CUTS[ETA_CUT_INDEX]
ETA_UNFOLDING_RANGE = (-ETA_CUT, ETA_CUT)
eta_cut_tag = f'{ETA_CUT:g}'.replace('.', 'p')
OUTPUT_DIR = Path(os.environ.get(
    'DIJET_UNFOLD2D_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis' / 'output' / 'unfold2D',
))
OUTPUT_TAG = f'{GENERATOR}_{DIRECTION}_unfold2D_jerDefExtra_to_{COMPARISON_TARGET}_eta_{eta_cut_tag}_iter_{N_ITERATIONS}'
OUTPUT_ROOT_FILE = OUTPUT_DIR / f'{OUTPUT_TAG}.root'
eta_cuts = ETA_CUTS
pt_ave_bins = as_pt_intervals(PT_AVE_BINS)

generator_label

In [ ]:
# Cell role: resolve input files and load the named ROOT objects.
# Interpretation: Loaded objects that outlive their file must be cloned and detached from ROOT directories.
# The preceding Markdown gives the equations and physics assumptions for this step.
from hist_analysis.python.histogram_io import load_histogram
n_eta_cuts = len(eta_cuts)
eta_idx = ETA_CUT_INDEX
keys = UnfoldingInputKeys(
    truth=f'hGenDijetPtEtaCM_{eta_idx}',
    measured=MEASURED_HISTOGRAM_TEMPLATE.format(eta_cut_index=eta_idx),
    response=RESPONSE_HISTOGRAM_TEMPLATE.format(eta_cut_index=eta_idx),
    miss=MISS_HISTOGRAM_TEMPLATE.format(eta_cut_index=eta_idx),
    fake=FAKE_HISTOGRAM_TEMPLATE.format(eta_cut_index=eta_idx),
    classification=CLASSIFICATION_HISTOGRAM_TEMPLATE.format(eta_cut_index=eta_idx),
)
inputs = load_unfolding_inputs(input_path, keys)
# Keep the legacy eta-cut-indexed containers used by the plotting cells.
genPtEtaCM = [None] * n_eta_cuts
refPtEtaCM = [None] * n_eta_cuts
recoPtEtaCM = [None] * n_eta_cuts
genPtEtaCMMiss = [None] * n_eta_cuts
recoPtEtaCMFake = [None] * n_eta_cuts
genPtEtaCMVsRecoPtEtaCM = [None] * n_eta_cuts
unfoldingPairClassification = [None] * n_eta_cuts
genPtEtaCM[eta_idx] = inputs.truth
refPtEtaCM[eta_idx] = load_histogram(str(input_path), f'hRefDijetPtEtaCM_{eta_idx}')
recoPtEtaCM[eta_idx] = inputs.measured
genPtEtaCMMiss[eta_idx] = inputs.miss
recoPtEtaCMFake[eta_idx] = inputs.fake
genPtEtaCMVsRecoPtEtaCM[eta_idx] = inputs.response
unfoldingPairClassification[eta_idx] = inputs.classification
print(input_path)
print(keys)

In [ ]:
# Cell role: draw the configured diagnostic figures and retain ROOT objects for display.
# Interpretation: Axis limits and logarithmic scales affect presentation only, not stored bin contents.
# The preceding Markdown gives the equations and physics assumptions for this step.
classification_canvas, classification = draw_unfolding_classification(
    unfoldingPairClassification[ETA_CUT_INDEX],
    annotations=(generator_label, f'|#eta_{{CM}}^{{jet}}| < {ETA_CUTS[ETA_CUT_INDEX]:g}', MEASURED_LABEL),
    output=OUTPUT_DIR / f'{OUTPUT_TAG}_pair_classification.pdf',
    save_png=SAVE_PNG, grid=DRAW_GRID,
    canvas_name='canvas_unfolding_classification_jerDefExtra',
)
classification_canvas

In [ ]:
# Cell role: draw the configured diagnostic figures and retain ROOT objects for display.
# Interpretation: Axis limits and logarithmic scales affect presentation only, not stored bin contents.
# The preceding Markdown gives the equations and physics assumptions for this step.
# Optional debugging example. Uncomment only when inspecting the source TH2.

# if not genPtEtaCM:
#     raise RuntimeError("genPtEtaCM is empty; load the ROOT file first")

# canvas_name = "canvas_gen_pt_eta_cm_0"
# existing_canvas = ROOT.gROOT.FindObject(canvas_name)
# if existing_canvas:
#     existing_canvas.Close()

# canvas_gen_dijet_pt_eta_cm_0 = ROOT.TCanvas(canvas_name, "cGenDijetPtEtaCM_0", 800, 800)
# canvas_gen_dijet_pt_eta_cm_0.cd()
# set_pad_style(ROOT.gPad, grid_x=False, grid_y=False)
# ROOT.gPad.SetRightMargin(DEFAULT_PLOT_STYLE.palette_right_margin)
# set_2d_style(genPtEtaCM[0])
# genPtEtaCM[0].Draw("COLZ")
# canvas_gen_dijet_pt_eta_cm_0.Modified()
# canvas_gen_dijet_pt_eta_cm_0.Update()
# canvas_gen_dijet_pt_eta_cm_0

In [ ]:
# Cell role: project multidimensional inputs into the configured analysis bins.
# Interpretation: Selections are applied before arithmetic so migrations and boundary bins remain explicit.
# The preceding Markdown gives the equations and physics assumptions for this step.
projection_options = {'pt_bins': pt_ave_bins, 'eta_range': ETA_UNFOLDING_RANGE}
def project_selected_eta(histogram, label):
    return project_eta_by_pt(histogram, name_prefix=f'{label}_{eta_idx}', **projection_options)

genEtaCM = [None] * n_eta_cuts
refEtaCM = [None] * n_eta_cuts
recoEtaCM = [None] * n_eta_cuts
genEtaCMMiss = [None] * n_eta_cuts
recoEtaCMFake = [None] * n_eta_cuts
gen2recoResponse = [None] * n_eta_cuts
genEtaCM[eta_idx] = project_selected_eta(genPtEtaCM[eta_idx], 'hGenDijetEtaCM')
refEtaCM[eta_idx] = project_selected_eta(refPtEtaCM[eta_idx], 'hRefDijetEtaCM')
recoEtaCM[eta_idx] = project_selected_eta(recoPtEtaCM[eta_idx], 'hRecoDijetEtaCM')
genEtaCMMiss[eta_idx] = project_selected_eta(genPtEtaCMMiss[eta_idx], 'hGenDijetEtaCMMiss')
recoEtaCMFake[eta_idx] = project_selected_eta(recoPtEtaCMFake[eta_idx], 'hRecoDijetEtaCMFake')
gen2recoResponse[eta_idx] = project_response_eta_blocks(
    genPtEtaCMVsRecoPtEtaCM[eta_idx], pt_ave_bins,
    name_prefix=f'hGen2RecoDijetEtaCM_{eta_idx}',
)

In [ ]:
# Cell role: project multidimensional inputs into the configured analysis bins.
# Interpretation: Selections are applied before arithmetic so migrations and boundary bins remain explicit.
# The preceding Markdown gives the equations and physics assumptions for this step.
# Optional debugging example. The standard workflow uses one canvas per pTave block.
# if not genEtaCM:
#     raise RuntimeError("genEtaCM is empty; run the projection cell first")

# eta_idx = 5  # choose eta-cut index here
# if eta_idx < 0 or eta_idx >= len(eta_cuts):
#     raise IndexError(f"eta_idx={eta_idx} is out of range for eta_cuts")

# canvas_name = f"canvas_genEtaCM_allPt_eta{eta_idx}"
# existing_canvas = ROOT.gROOT.FindObject(canvas_name)
# if existing_canvas:
#     existing_canvas.Close()

# canvas_genEtaCM_allPt = ROOT.TCanvas(canvas_name, f"Gen etaCM projections (eta idx {eta_idx})", 800, 800)
# canvas_genEtaCM_allPt.cd()
# set_pad_style(ROOT.gPad, grid_x=False, grid_y=False)

# gen_eta_overlays = []
# max_val = 0.0
# for pt_idx in range(len(pt_ave_bins) - 1):
#     hist = genEtaCM[eta_idx][pt_idx].Clone(f"hGenEtaCM_overlay_eta{eta_idx}_pt{pt_idx}")
#     hist.SetDirectory(0)
#     set_unfolding_1d_style(hist, 'gen')
#     integral = hist.Integral()
#     if integral > 0:
#         hist.Scale(1.0 / integral)
#     hist.SetTitle(";#eta_{CM};Normalized entries")
#     draw_opt = "E1" if pt_idx == 0 else "E1 SAME"
#     hist.Draw(draw_opt)
#     gen_eta_overlays.append(hist)
#     max_val = max(max_val, hist.GetMaximum())

# legend = ROOT.TLegend(0.6, 0.75, 0.88, 0.88)
# set_legend_style(legend)
# legend.SetTextFont(42)
# legend.SetTextSize(0.03)
# for pt_idx in range(len(pt_ave_bins) - 1):
#     label = f"{pt_ave_bins[pt_idx]} < p_{{T}}^{{ave}} < {pt_ave_bins[pt_idx + 1]} GeV"
#     legend.AddEntry(gen_eta_overlays[pt_idx], label, "p")
# legend.Draw()

# text = ROOT.TLatex()
# text.SetNDC(True)
# text.SetTextFont(42)
# text.SetTextSize(0.04)
# text.DrawLatex(0.16, 0.92, f"Gen #eta_{{CM}} projections, eta-cut index = {eta_idx}")

# canvas_genEtaCM_allPt.Modified()
# canvas_genEtaCM_allPt.Update()
# canvas_genEtaCM_allPt

In [ ]:
# Cell role: project multidimensional inputs into the configured analysis bins.
# Interpretation: Selections are applied before arithmetic so migrations and boundary bins remain explicit.
# The preceding Markdown gives the equations and physics assumptions for this step.
eta_idx = ETA_CUT_INDEX
projection_response_canvases = []
for pt_bin_idx, (pt_low, pt_high) in enumerate(pt_ave_bins):
    spectra = {
        'gen': ('Gen', genEtaCM[eta_idx][pt_bin_idx]),
        'reco': (MEASURED_LABEL, recoEtaCM[eta_idx][pt_bin_idx]),
        'miss': ('Miss', genEtaCMMiss[eta_idx][pt_bin_idx]),
        'fake': ('Fake', recoEtaCMFake[eta_idx][pt_bin_idx]),
    }
    if COMPARISON_TARGET == 'ref':
        spectra = {'gen': spectra['gen'], 'ref': ('Ref comparison target', refEtaCM[eta_idx][pt_bin_idx]), **{k: v for k, v in spectra.items() if k != 'gen'}}
    canvas = draw_projection_response(
        spectra, gen2recoResponse[eta_idx][pt_bin_idx],
        eta_range=ETA_UNFOLDING_RANGE,
        response_titles=(f'{MEASURED_LABEL} #eta_{{CM}}', 'Gen #eta_{CM}'),
        annotations=(generator_label, f'{pt_low:g} < p_{{T}}^{{ave}} < {pt_high:g} GeV', f'|#eta_{{CM}}| < {eta_cuts[eta_idx]}', DIJET_DELTA_PHI_SELECTION_LABEL),
        plot_miss_and_fakes=PLOT_MISS_AND_FAKES,
        output=OUTPUT_DIR / f'{OUTPUT_TAG}_projection_response_pt_{pt_low:g}_{pt_high:g}.pdf',
        save_png=SAVE_PNG, grid=DRAW_GRID, canvas_name=f'canvas_projection_response_ptBin{pt_bin_idx}',
    )
    projection_response_canvases.append(canvas)
projection_response_canvases

In [ ]:
# Cell role: project multidimensional inputs into the configured analysis bins.
# Interpretation: Selections are applied before arithmetic so migrations and boundary bins remain explicit.
# The preceding Markdown gives the equations and physics assumptions for this step.
eta_idx = ETA_CUT_INDEX
hGenTruthEtaCM, layout = flatten_pt_eta_projections(genEtaCM[eta_idx], name='hGenTruthEtaCM', pt_bins=pt_ave_bins)
hRefComparisonEtaCM, _ = flatten_pt_eta_projections(refEtaCM[eta_idx], name='hRefComparisonEtaCM', layout=layout)
hRecoMeasuredEtaCM, _ = flatten_pt_eta_projections(recoEtaCM[eta_idx], name='hRecoMeasuredEtaCM', layout=layout)
hGenTruthEtaCMMiss, _ = flatten_pt_eta_projections(genEtaCMMiss[eta_idx], name='hGenTruthEtaCMMiss', layout=layout)
hRecoMeasuredEtaCMFake, _ = flatten_pt_eta_projections(recoEtaCMFake[eta_idx], name='hRecoMeasuredEtaCMFake', layout=layout)
hResponseEtaCM, _ = flatten_sparse_response(genPtEtaCMVsRecoPtEtaCM[eta_idx], pt_ave_bins, name='hResponseEtaCM', layout=layout, eta_range=ETA_UNFOLDING_RANGE)
nPtSelections, nEtaBins, nGlobalBins = layout.n_pt_bins, layout.n_eta_bins, layout.n_global_bins
comparisonTargetLabel = 'Gen' if COMPARISON_TARGET == 'gen' else 'Ref'
hComparisonTargetEtaCM = hGenTruthEtaCM if COMPARISON_TARGET == 'gen' else hRefComparisonEtaCM
print(f'nPtSelections={nPtSelections}, nEtaBins={nEtaBins}, nGlobalBins={nGlobalBins}')

In [ ]:
# Cell role: project multidimensional inputs into the configured analysis bins.
# Interpretation: Selections are applied before arithmetic so migrations and boundary bins remain explicit.
# The preceding Markdown gives the equations and physics assumptions for this step.
flattened_spectra = {
    'gen': ('Gen', hGenTruthEtaCM),
    'reco': (MEASURED_LABEL, hRecoMeasuredEtaCM),
    'miss': ('Miss', hGenTruthEtaCMMiss),
    'fake': ('Fake', hRecoMeasuredEtaCMFake),
}
if COMPARISON_TARGET == 'ref':
    flattened_spectra = {'gen': flattened_spectra['gen'], 'ref': ('Ref comparison target', hRefComparisonEtaCM), **{k: v for k, v in flattened_spectra.items() if k != 'gen'}}
canvas_flattened = draw_flattened_response(
    flattened_spectra, hResponseEtaCM, annotations=(generator_label, f'|#eta_{{CM}}| < {eta_cuts[eta_idx]}'),
    plot_miss_and_fakes=PLOT_MISS_AND_FAKES, output=OUTPUT_DIR / f'{OUTPUT_TAG}_flattened_response.pdf',
    save_png=SAVE_PNG, grid=DRAW_GRID, canvas_name='canvas_flattened',
)
canvas_flattened

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
diagnostics = calculate_response_diagnostics(
    hResponseEtaCM, hGenTruthEtaCM, hRecoMeasuredEtaCM,
    explicit_miss=hGenTruthEtaCMMiss, explicit_fake=hRecoMeasuredEtaCMFake,
)
hMatchedTruthEtaCM = diagnostics.matched_truth
hMatchedRecoEtaCM = diagnostics.matched_measured
hEffectiveMissEtaCM = diagnostics.effective_miss
hEffectiveFakeEtaCM = diagnostics.effective_fake
hBoundaryMissEtaCM = diagnostics.boundary_miss
hBoundaryFakeEtaCM = diagnostics.boundary_fake
response_accounting = validate_response_accounting(
    hResponseEtaCM, hGenTruthEtaCM, hRecoMeasuredEtaCM, diagnostics,
)
print('Response accounting:')
print(f'  max truth residual: {response_accounting.max_truth_residual:.3g}')
print(f'  max measured residual: {response_accounting.max_measured_residual:.3g}')
print(f'  global efficiency: {response_accounting.global_efficiency:.4f}')
print(f'  global fake fraction: {response_accounting.global_fake_fraction:.4f}')
print(f'  response sparsity: {response_accounting.response_sparsity:.4f}')
print(f'  zero-efficiency truth bins: {response_accounting.zero_efficiency_truth_bins}')
print(f'  empty measured bins: {response_accounting.empty_measured_bins}')

# Reference calculation: RooUnfold handles the effective fakes and misses.
inclusive_response_bundle = build_roounfold_response(
    RooUnfold, hGenTruthEtaCM, hRecoMeasuredEtaCM, hResponseEtaCM,
    diagnostics=diagnostics, scale=RESPONSE_SCALE, require_fakes=REQUIRE_FAKES,
)
inclusive_unfolding_result = unfold_bayes(
    RooUnfold, inclusive_response_bundle, hRecoMeasuredEtaCM, iterations=N_ITERATIONS,
    handle_fakes=HANDLE_FAKES, name='hUnfoldedInclusiveResponseEtaCM',
)
hUnfoldedInclusiveResponseEtaCM = inclusive_unfolding_result.histogram

# New calculation: purity -> matched migration unfolding -> efficiency.
factorized_inputs = prepare_factorized_corrections(
    hRecoMeasuredEtaCM, hRecoMeasuredEtaCM, hMatchedRecoEtaCM,
    hGenTruthEtaCM, hMatchedTruthEtaCM,
)
hRecoSignalEtaCM = factorized_inputs.measured_signal
hRecoPurityEtaCM = factorized_inputs.purity
hTruthEfficiencyEtaCM = factorized_inputs.efficiency
matched_response_bundle = build_roounfold_response(
    RooUnfold, hMatchedTruthEtaCM, hMatchedRecoEtaCM, hResponseEtaCM,
    scale=RESPONSE_SCALE, require_fakes=False, name='responseMatchedEtaCM',
)
matched_unfolding_result = unfold_bayes(
    RooUnfold, matched_response_bundle, hRecoSignalEtaCM, iterations=N_ITERATIONS,
    handle_fakes=False, name='hUnfoldedMatchedEtaCM',
)
hUnfoldedMatchedEtaCM = matched_unfolding_result.histogram
hUnfoldedEtaCM, covariance_matrix = apply_efficiency_correction(
    hUnfoldedMatchedEtaCM, matched_unfolding_result.covariance,
    hTruthEfficiencyEtaCM, name='hUnfoldedEtaCM',
)
unfold = matched_unfolding_result.algorithm
response_bundle = matched_response_bundle
response = matched_response_bundle.response
print(f'Explicit misses: {hGenTruthEtaCMMiss.Integral():.6g}; effective: {hEffectiveMissEtaCM.Integral():.6g}')
print(f'Explicit fakes: {hRecoMeasuredEtaCMFake.Integral():.6g}; effective: {hEffectiveFakeEtaCM.Integral():.6g}')

response_matrix_canvas = draw_response_matrix(
    hResponseEtaCM,
    annotations=(generator_label, f'|#eta_{{CM}}| < {ETA_CUT:g}', f'{layout.n_pt_bins} p_{{T}}^{{ave}} intervals'),
    output=OUTPUT_DIR / f'{OUTPUT_TAG}_response_matrix.pdf',
    save_png=SAVE_PNG, grid=False, log_z=True,
    canvas_name='canvas_response_matrix',
)
response_components_canvas = draw_response_components(
    explicit_miss=hGenTruthEtaCMMiss, boundary_miss=hBoundaryMissEtaCM, effective_miss=hEffectiveMissEtaCM,
    explicit_fake=hRecoMeasuredEtaCMFake, boundary_fake=hBoundaryFakeEtaCM, effective_fake=hEffectiveFakeEtaCM,
    annotations=(generator_label, f'|#eta_{{CM}}| < {ETA_CUT:g}'),
    output=OUTPUT_DIR / f'{OUTPUT_TAG}_response_components.pdf',
    save_png=SAVE_PNG, grid=DRAW_GRID,
)

In [ ]:
# Cell role: project multidimensional inputs into the configured analysis bins.
# Interpretation: Selections are applied before arithmetic so migrations and boundary bins remain explicit.
# The preceding Markdown gives the equations and physics assumptions for this step.
canvas_unfolded, hMeasuredToTruth, hUnfoldedToTruth = draw_unfolding_closure(
    hComparisonTargetEtaCM, hRecoMeasuredEtaCM, hUnfoldedEtaCM,
    target_label=comparisonTargetLabel, target_role=COMPARISON_TARGET, measured_label=MEASURED_LABEL,
    x_title='global #eta_{CM} bin', ratio_range=RATIO_Y_RANGE,
    annotations=(generator_label, f'|#eta_{{CM}}| < {ETA_CUT:g}', f'Bayesian iterations: {N_ITERATIONS}'),
    output=OUTPUT_DIR / f'{OUTPUT_TAG}_closure.pdf', save_png=SAVE_PNG,
    grid=DRAW_GRID, canvas_name='canvas_unfolded', ratio_name_prefix='hFlattenedClosure',
)
display(response_matrix_canvas)
display(response_components_canvas)
display(canvas_unfolded)

# The two methods need not be identical after a finite number of Bayesian
# iterations, so compare them directly as a validation diagnostic.
canvas_method_comparison, method_comparison_ratios = draw_closure(
    {comparisonTargetLabel: hComparisonTargetEtaCM,
     'Factorized': hUnfoldedEtaCM,
     'RooUnfold fakes/misses': hUnfoldedInclusiveResponseEtaCM},
    nominal=comparisonTargetLabel, title='', x_title='global #eta_{CM} bin',
    y_title='Entries', ratio_range=RATIO_Y_RANGE, draw_nominal_ratio=False,
    annotations=(generator_label, f'|#eta_{{CM}}| < {ETA_CUT:g}',
                 f'Bayesian iterations: {N_ITERATIONS}'),
    output=OUTPUT_DIR / f'{OUTPUT_TAG}_method_comparison.pdf',
    save_png=SAVE_PNG, grid=DRAW_GRID, canvas_name='canvas_method_comparison',
)
hFactorizedToTarget = method_comparison_ratios['Factorized']
hFactorizedToTarget.SetName('hFactorizedToTarget')
hInclusiveResponseToTarget = method_comparison_ratios['RooUnfold fakes/misses']
hInclusiveResponseToTarget.SetName('hInclusiveResponseToTarget')
truth_closure_metrics = calculate_closure_metrics(
    hUnfoldedEtaCM, hComparisonTargetEtaCM,
)
print('Factorized truth closure:')
print(f'  integral ratio = {truth_closure_metrics.integral_ratio:.6f}')
print(f'  yield-weighted L1 difference = {truth_closure_metrics.relative_l1:.2%}')
print(f'  mean relative difference in {truth_closure_metrics.compared_bins} populated bins = ' 
      f'{truth_closure_metrics.mean_absolute_relative:.2%}')
canvas_method_comparison

## Unfolded eta distributions in all pTave intervals

Extract every pTave block from the flattened unfolded histogram. For each interval, compare unfolded and eta-dependent JER-default reco eta distributions with the configured Gen or Ref comparison target. The response and Bayesian prior remain Gen-based for both choices.

This notebook reuses the response-training sample as pseudo-data. Its purity-corrected reco spectrum and truth prior are therefore algebraically consistent with the response, so closure should be unity to numerical precision. This is a pipeline identity test, not an independent statistical validation.

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
target_by_pt = genEtaCM[eta_idx] if COMPARISON_TARGET == 'gen' else refEtaCM[eta_idx]
(hUnfoldedEtaCMByPt, hRecoToComparisonEtaCMByPt,
 hUnfoldedToComparisonEtaCMByPt, unfolded_eta_canvases) = draw_unfolding_closure_by_pt(
    hUnfoldedEtaCM, target_by_pt, recoEtaCM[eta_idx], layout,
    output_dir=OUTPUT_DIR, output_tag=OUTPUT_TAG, target_label=comparisonTargetLabel,
    target_role=COMPARISON_TARGET, measured_label=MEASURED_LABEL,
    eta_range=ETA_UNFOLDING_RANGE, ratio_range=RATIO_Y_RANGE,
    annotation_prefix=(generator_label, f'Bayesian iterations: {N_ITERATIONS}'),
    save_png=SAVE_PNG, grid=DRAW_GRID,
)
hGenEtaCMByPt = [hist.Clone(f'hGenEtaCM_ptBin{i}') for i, hist in enumerate(genEtaCM[eta_idx])]
hRefEtaCMByPt = [hist.Clone(f'hRefEtaCM_ptBin{i}') for i, hist in enumerate(refEtaCM[eta_idx])]
hRecoEtaCMByPt = [hist.Clone(f'hRecoEtaCM_ptBin{i}') for i, hist in enumerate(recoEtaCM[eta_idx])]
unfolded_eta_canvases

## Forward-folded closure

Apply the inclusive trained detector response to the efficiency-corrected unfolded distribution. Its matrix columns reapply efficiency and migration, producing matched reco. Then restore fakes using the training fake-to-matched-reco ratio, which is equivalent to dividing by purity. Compare the inclusive prediction with measured reco in flattened global bins and separately in every $p_T^{ave}$ interval. Ratios use standard independent-error propagation.

In [ ]:
# Cell role: project multidimensional inputs into the configured analysis bins.
# Interpretation: Selections are applied before arithmetic so migrations and boundary bins remain explicit.
# The preceding Markdown gives the equations and physics assumptions for this step.
forward_fold_result = forward_fold_truth(
    inclusive_response_bundle, hUnfoldedEtaCM, diagnostics=diagnostics,
    add_fakes=FORWARD_FOLD_ADD_FAKES, name='hForwardFoldedUnfoldedEtaCM',
)
hForwardFoldedEtaCM = forward_fold_result.histogram
print(f'ApplyToTruth included fakes: {forward_fold_result.apply_to_truth_includes_fakes}')

canvas_forward_folded, forward_folded_flattened_ratios = draw_closure(
    {'Reco': hRecoMeasuredEtaCM, 'Forward-folded': hForwardFoldedEtaCM},
    nominal='Reco', title='', x_title='global #eta_{CM} bin', y_title='Entries',
    ratio_range=RATIO_Y_RANGE, draw_nominal_ratio=False,
    annotations=(generator_label, f'|#eta_{{CM}}| < {ETA_CUT:g}', f'Bayesian iterations: {N_ITERATIONS}'),
    output=OUTPUT_DIR / f'{OUTPUT_TAG}_forward_folded_closure.pdf',
    save_png=SAVE_PNG, grid=DRAW_GRID,
    canvas_name='canvas_forward_folded_closure',
)
hForwardFoldedToReco = forward_folded_flattened_ratios['Forward-folded']
hForwardFoldedToReco.SetName('hForwardFoldedToReco')
refold_closure_metrics = calculate_closure_metrics(
    hForwardFoldedEtaCM, hRecoMeasuredEtaCM,
)
print('Forward-folded reco closure:')
print(f'  integral ratio = {refold_closure_metrics.integral_ratio:.6f}')
print(f'  yield-weighted L1 difference = {refold_closure_metrics.relative_l1:.2%}')
print(f'  mean relative difference in {refold_closure_metrics.compared_bins} populated bins = ' 
      f'{refold_closure_metrics.mean_absolute_relative:.2%}')

hForwardFoldedEtaCMByPt = unflatten_to_eta_projections(
    hForwardFoldedEtaCM, recoEtaCM[eta_idx], layout,
    name_prefix='hForwardFoldedEtaCM',
)
forward_folded_canvases_by_pt = []
hForwardFoldedToRecoByPt = []
for pt_bin_idx, ((pt_low, pt_high), reco_hist, folded_hist) in enumerate(
    zip(pt_ave_bins, recoEtaCM[eta_idx], hForwardFoldedEtaCMByPt)
):
    canvas, ratios = draw_closure(
        {'Reco': reco_hist, 'Forward-folded': folded_hist},
        nominal='Reco', title='', x_title='#eta_{CM}', y_title='Entries',
        ratio_range=RATIO_Y_RANGE, draw_nominal_ratio=False,
        x_range=ETA_UNFOLDING_RANGE,
        annotations=(generator_label, f'{pt_low:g} < p_{{T}}^{{ave}} < {pt_high:g} GeV', f'Bayesian iterations: {N_ITERATIONS}'),
        output=OUTPUT_DIR / f'{OUTPUT_TAG}_forward_folded_closure_pt_{pt_low:g}_{pt_high:g}.pdf',
        save_png=SAVE_PNG, grid=DRAW_GRID,
        canvas_name=f'canvas_forward_folded_closure_pt{pt_bin_idx}',
    )
    ratio = ratios['Forward-folded']
    ratio.SetName(f'hForwardFoldedToReco_ptBin{pt_bin_idx}')
    forward_folded_canvases_by_pt.append(canvas)
    hForwardFoldedToRecoByPt.append(ratio)

display(canvas_forward_folded)
forward_folded_canvases_by_pt

## Save unfolding output

Write the flattened spectra, response diagnostics, unfolded result, ratios, covariance matrix, and configuration metadata to `hist_analysis/output/unfold2D/`.

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
output_histograms = (
    hGenTruthEtaCM, hRefComparisonEtaCM, hRecoMeasuredEtaCM, hGenTruthEtaCMMiss, hRecoMeasuredEtaCMFake, hResponseEtaCM,
    hMatchedTruthEtaCM, hMatchedRecoEtaCM, hEffectiveMissEtaCM, hEffectiveFakeEtaCM, hBoundaryMissEtaCM, hBoundaryFakeEtaCM,
    hRecoSignalEtaCM, hRecoPurityEtaCM, hTruthEfficiencyEtaCM,
    hUnfoldedMatchedEtaCM, hUnfoldedEtaCM, hUnfoldedInclusiveResponseEtaCM,
    hMeasuredToTruth, hUnfoldedToTruth, hFactorizedToTarget, hInclusiveResponseToTarget,
    *hGenEtaCMByPt, *hRefEtaCMByPt, *hRecoEtaCMByPt,
    *hUnfoldedEtaCMByPt, *hRecoToComparisonEtaCMByPt, *hUnfoldedToComparisonEtaCMByPt,
    forward_fold_result.folded_signal, forward_fold_result.fake, hForwardFoldedEtaCM, hForwardFoldedToReco,
    *hForwardFoldedEtaCMByPt, *hForwardFoldedToRecoByPt,
)
write_unfolding_output(
    OUTPUT_ROOT_FILE, histograms=output_histograms, covariance=covariance_matrix, response=response,
    metadata={'generator': GENERATOR, 'direction': DIRECTION, 'eta_cut': ETA_CUT,
              'eta_unfolding_range': ETA_UNFOLDING_RANGE, 'pt_ave_bins': pt_ave_bins,
              'ptave_bin_set': PTAVE_BIN_SET, 'iterations': N_ITERATIONS, 'comparison_target': COMPARISON_TARGET,
              'plot_miss_and_fakes': PLOT_MISS_AND_FAKES, 'response_scale': RESPONSE_SCALE,
              'unfolding_method': 'factorized_purity_migration_efficiency',
              'require_fakes_reference': REQUIRE_FAKES, 'handle_fakes_reference': HANDLE_FAKES,
              'forward_fold_add_fakes': FORWARD_FOLD_ADD_FAKES,
              'apply_to_truth_includes_fakes': forward_fold_result.apply_to_truth_includes_fakes,
              'response_sparsity': response_accounting.response_sparsity,
              'global_efficiency': response_accounting.global_efficiency,
              'global_fake_fraction': response_accounting.global_fake_fraction,
              'truth_closure_relative_l1': truth_closure_metrics.relative_l1,
              'refold_closure_relative_l1': refold_closure_metrics.relative_l1},
)
print(f'Wrote unfolding output to {OUTPUT_ROOT_FILE}')